Reading File

In [9]:
WFDB,CSV,MORTARA=0,0,1
n=4
file_path=f'Pats/pat{n}/pat{n}.txt'
# file_path='more/mitdb/213'
# WFDB  records
import wfdb 
if WFDB:
    record = wfdb.rdrecord(file_path)
    fs=record.fs
    print(f'fs={fs}')
    sig=record.p_signal[:,0]

# CSV records
import pandas as pd
if CSV:
    csv_df = pd.read_csv(file_path)
    if 'time' in csv_df.columns:
        csv_time = csv_df['time'].values
        csv_signals = csv_df.drop(columns=['time']).values
    else:
        csv_signals = csv_df.values
    csv_fs = 250 

# Mortara Records 
from functions import dat_reader
if MORTARA:
    fs=180
    lead='DataII'
    sig,_=dat_reader(file_path,fs,f'pat{n}',lead)
    sig=sig*-1
    # import matplotlib.pyplot as plt 
    # %matplotlib tk
    # plt.plot(sig)
    # plt.show()

In [ ]:

sig=sig[:100000]
print(len(sig)/(180*3600))

0.15432098765432098


In [3]:
import matplotlib.pyplot as plt 
%matplotlib tk
plt.plot(sig)
plt.show()

Preprocess

In [11]:
from features import bessel_bandpass_filter
# Noise+baseline wander elimination (0.5-40Hz)
print('Filtering the signal...')
sig=bessel_bandpass_filter(sig,fs) 
print('Finished Filtering!')
import numpy as np 
print("Resampling...")
# def resample_linear(x, n):
#     x_old = np.linspace(0, 1, len(x))
#     x_new = np.linspace(0, 1, n)
#     return np.interp(x_new, x_old, x)
# nl=int(360*len(sig)/
# fs)
# sig=resample_linear(sig,nl)
# print("Finished Resampling!")
# fs=360


Filtering the signal...
Finished Filtering!
Resampling...


In [75]:
print(len(sig))

1000000


R Detection

In [13]:
from functions import r_detection
r_peaks=r_detection(sig,fs)
print(len(r_peaks))

fs is 180 and signal length is 0.15432098765432098 hours
chunks=9 len_sig=100000
653


In [46]:
# plt.plot(sig)
# plt.scatter(r_peaks,sig[r_peaks],color='red')
# plt.close()
# import importlib
# import functions
# importlib.reload(functions)
# from functions import r_detection
# sig=sig[:100000]
# print(len(sig))
# r_peaks=r_detection(sig,fs=180)

Beat Prediction (CNN Network)


In [12]:
import importlib
import hierarchical_qrs_classifier
importlib.reload(hierarchical_qrs_classifier)
from hierarchical_qrs_classifier import HierarchicalQRSClassifier
classifier=HierarchicalQRSClassifier()
print("Beat Annotating...")
result=classifier.predict(sig,r_peaks,fs)
r=result["abnormal_probability"]
threshold=0.0000001
rest=[]
good=[]
for i in range(len(r)):
    if (r[i]>threshold):
        rest.append(r_peaks[i])
    else:
        good.append(r_peaks[i])
print("Finished Annotating!")
rest=np.array(rest)
print(f'{len(rest)} of {len(r_peaks)} detected abnormal!')

Beat Annotating...
Finished Annotating!
6662 of 6989 detected abnormal!


In [13]:
good=np.array(good)
print(len(good))
pre=int(fs/5) #200ms before
post=int(fs/4) #250ms after
for i in good :
    beat=sig[i-pre:i+post]
    plt.plot(beat)
plt.show()

327


C:\Users\Artin\AppData\Local\Temp\ipykernel_17344\779548963.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Cluster Abnormal Beats(HDBSCAN)

In [88]:
import fp_inference
import importlib
print
importlib.reload(fp_inference)
record_name='Mortara/pat90'
from fp_inference import infer_beats
print(len(r_peaks))
res = infer_beats(sig, rest, record_name, fs)

6989
max_beats=300000
Mortara/pat90: 6661 usable beats of 6662 given
6661 beats clustered in 20-D PCA space, no reference labels (ARI / purity not computed)
mcs=3

method    clusters  noise  silhouette     ARI  purity
kmeans           2      0       0.260     -       -  
hdbscan          2   1748       0.362     -       -  

Saved to c:\Users\Artin\Desktop\ecg_hierarchical_v4_pipeline\inference\Mortara\pat90


In [19]:
# import numpy as np
# from fp_inference_raw import infer_beats
import importlib
# res = infer_beats(sig, r_peaks, "mortara/pat4", fs=180, merge_amp_tol=0.2)
import fp_hdbscan_only
importlib.reload(fp_hdbscan_only)
from fp_hdbscan_only import hdbscan_beats

res = hdbscan_beats(sig, r_peaks, "mortara/4", fs=fs)                    # one HDBSCAN on all beats
# res = hdbscan_beats(sig, r_peaks, "mitdb/100", fs=fs, mode="chunks")     # HDBSCAN per block of 20,000 beats             # with amplitude tolerance

mortara/4: fs=180 Hz, window 36 + 45 = 81 samples, 71638 usable beats of 71639 given


c:\Users\Artin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


block 1/1: beats 0-71637, 34 PCs, 2 clusters, 9062 noise
hdbscan: 2 clusters, 9062 noise beats, sizes [62572, 4]

Saved to c:\Users\Artin\Desktop\ecg_hierarchical_v4_pipeline\inference\mortara\4


In [4]:
WFDB,CSV,MORTARA=0,0,1
for i in range(5):
    n=i+1
    if(n==3):
        continue
    file_path=f'Pats/pat{n}/pat{n}.txt'
    # file_path='more/mitdb/213'
    # WFDB  records
    import wfdb 
    if WFDB:
        record = wfdb.rdrecord(file_path)
        fs=record.fs
        print(f'fs={fs}')
        sig=record.p_signal[:,0]

    # CSV records
    import pandas as pd
    if CSV:
        csv_df = pd.read_csv(file_path)
        if 'time' in csv_df.columns:
            csv_time = csv_df['time'].values
            csv_signals = csv_df.drop(columns=['time']).values
        else:
            csv_signals = csv_df.values
        csv_fs = 250 

    # Mortara Records 
    a=-1
    from functions import dat_reader
    if MORTARA:
        print(f"processing pat{n}")
        fs=180
        lead='DataII'
        sig,_=dat_reader(file_path,fs,f'pat{n}',lead)
        if(n==1):
            a=1
        else:
            a=-1
        sig=sig*a
        from features import bessel_bandpass_filter
        # Noise+baseline wander elimination (0.5-40Hz)
        print('Filtering the signal...')
        sig=bessel_bandpass_filter(sig,fs) 
        print('Finished Filtering!')
        import numpy as np 
        from functions import r_detection
        r_peaks=r_detection(sig,fs)
        import importlib
        # res = infer_beats(sig, r_peaks, "mortara/pat4", fs=180, merge_amp_tol=0.2)
        import fp_hdbscan_only
        importlib.reload(fp_hdbscan_only)
        from fp_hdbscan_only import hdbscan_beats

        res = hdbscan_beats(sig, r_peaks, f"mortara/pat_{n}", fs=fs)    
        # import matplotlib.pyplot as plt 
        # %matplotlib tk
        # plt.plot(sig)
        # plt.show()

processing pat1
Filtering the signal...
Finished Filtering!
fs is 180 and signal length is 47.99166666666667 hours
chunks=2879 len_sig=31098600
mortara/pat_1: fs=180 Hz, window 36 + 45 = 81 samples, 222049 usable beats of 222049 given
HDBSCAN on all 222049 beats at once: this can take many minutes


c:\Users\Artin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


block 1/1: beats 0-222048, 24 PCs, 3310 clusters, 160498 noise
hdbscan: 3310 clusters, 160498 noise beats, sizes [47322, 44, 27, 25, 20, 20, 19, 18, 17, 17, 16, 16, 15, 15, 15] ...

Saved to c:\Users\Artin\Desktop\ecg_hierarchical_v4_pipeline\inference\mortara\pat_1
processing pat2
Filtering the signal...
Finished Filtering!
fs is 180 and signal length is 47.99166666666667 hours
chunks=2879 len_sig=31098600


c:\Users\Artin\AppData\Local\Programs\Python\Python313\Lib\site-packages\vg_beat_detectors\vg_detector.py:221: RuntimeWarning: invalid value encountered in divide
  w_new = abs(Av / np.linalg.norm(Av))


mortara/pat_2: fs=180 Hz, window 36 + 45 = 81 samples, 187784 usable beats of 187785 given
HDBSCAN on all 187784 beats at once: this can take many minutes


c:\Users\Artin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


block 1/1: beats 0-187783, 21 PCs, 1976 clusters, 58868 noise
hdbscan: 1976 clusters, 58868 noise beats, sizes [100501, 1318, 1159, 1075, 906, 890, 725, 576, 511, 458, 453, 453, 402, 377, 376] ...

Saved to c:\Users\Artin\Desktop\ecg_hierarchical_v4_pipeline\inference\mortara\pat_2
processing pat4
Filtering the signal...
Finished Filtering!
fs is 180 and signal length is 23.995833333333334 hours
chunks=1439 len_sig=15549300
mortara/pat_4: fs=180 Hz, window 36 + 45 = 81 samples, 108842 usable beats of 108843 given
HDBSCAN on all 108842 beats at once: this can take many minutes


c:\Users\Artin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


block 1/1: beats 0-108841, 13 PCs, 6 clusters, 125 noise
hdbscan: 6 clusters, 125 noise beats, sizes [99438, 7501, 1765, 6, 4, 3]

Saved to c:\Users\Artin\Desktop\ecg_hierarchical_v4_pipeline\inference\mortara\pat_4
processing pat5
Filtering the signal...
Finished Filtering!
fs is 180 and signal length is 47.99166666666667 hours
chunks=2879 len_sig=31098600
mortara/pat_5: fs=180 Hz, window 36 + 45 = 81 samples, 244529 usable beats of 244530 given
HDBSCAN on all 244529 beats at once: this can take many minutes


c:\Users\Artin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


block 1/1: beats 0-244528, 16 PCs, 6129 clusters, 216848 noise
hdbscan: 6129 clusters, 216848 noise beats, sizes [62, 50, 43, 38, 37, 37, 34, 31, 28, 26, 25, 24, 23, 23, 22] ...

Saved to c:\Users\Artin\Desktop\ecg_hierarchical_v4_pipeline\inference\mortara\pat_5


Cluster The Noises(Correlation)

In [ ]:
import numpy as np
d = np.load("inference/Mortara/pat/clusters/hdbscan/noise/noise_rpeaks.npz")
peaks = d["r_peaks"]  
import correlation_clustering
importlib.reload(correlation_clustering)
from correlation_clustering import cluster_beats_by_correlation

labels, valid_peaks = cluster_beats_by_correlation(
    signal=sig, 
    r_peaks=peaks, 
    output_folder='inference/Mortara/Pat90/clusters/hdbscan/noise', 
    threshold=0.90,
    pre=72,  # defaults to 72 samples before peak
    post=90  # defaults to 90 samples after peak
)

: 

In [6]:
import os
import shutil
from pathlib import Path
base='inference/Mortara/pat_2/clusters/hdbscan'
def consolidate_cluster_images(hdbscan_dir=base):
    """
    Extracts all cluster_n.png files from their respective subfolders 
    and copies them into a single 'clusters_all_in_one' folder, 
    ignoring the noise cluster.
    """
    base_path = Path(hdbscan_dir)
    
    # Create the target directory if it doesn't exist
    target_dir = base_path / "clusters_all_in_one"
    target_dir.mkdir(parents=True, exist_ok=True)
    
    # Iterate through all items in the HDBSCAN directory
    for item in base_path.iterdir():
        # Check if it's a directory and not our target directory
        if item.is_dir() and item.name != "clusters_all_in_one":
            
            # HDBSCAN typically labels noise as -1. We skip '-1' or explicitly named 'noise' folders
            if "-1" in item.name or "noise" in item.name.lower():
                print(f"Skipping noise folder: {item.name}")
                continue
                
            # Construct the expected image file path (e.g., cluster_0/cluster_0.png)
            expected_image_name = f"{item.name}.png"
            image_path = item / expected_image_name
            
            # If the image exists, copy it to the all_in_one folder
            if image_path.exists():
                destination_path = target_dir / expected_image_name
                shutil.copy2(image_path, destination_path)
                print(f"Copied: {expected_image_name}")
            else:
                print(f"Image not found in: {item.name}")

consolidate_cluster_images(base)
# To run it in your notebook:
# consolidate_cluster_images(hdbscan_dir="path/to/your/HDBSCAN")

Copied: cluster1.png
Copied: cluster10.png
Copied: cluster100.png
Copied: cluster1000.png
Copied: cluster1001.png
Copied: cluster1002.png
Copied: cluster1003.png
Copied: cluster1004.png
Copied: cluster1005.png
Copied: cluster1006.png
Copied: cluster1007.png
Copied: cluster1008.png
Copied: cluster1009.png
Copied: cluster101.png
Copied: cluster1010.png
Copied: cluster1011.png
Copied: cluster1012.png
Copied: cluster1013.png
Copied: cluster1014.png
Copied: cluster1015.png
Copied: cluster1016.png
Copied: cluster1017.png
Copied: cluster1018.png
Copied: cluster1019.png
Copied: cluster102.png
Copied: cluster1020.png
Copied: cluster1021.png
Copied: cluster1022.png
Copied: cluster1023.png
Copied: cluster1024.png
Copied: cluster1025.png
Copied: cluster1026.png
Copied: cluster1027.png
Copied: cluster1028.png
Copied: cluster1029.png
Copied: cluster103.png
Copied: cluster1030.png
Copied: cluster1031.png
Copied: cluster1032.png
Copied: cluster1033.png
Copied: cluster1034.png
Copied: cluster1035.png
C

In [59]:
import fp_inference
import importlib
d = np.load("inference/Mortara/pat8/clusters/hdbscan/cluster1/cluster1_rpeaks.npz")
peaks = d["r_peaks"]  
print(type(peaks))
importlib.reload(fp_inference)
record_name='Mortara/pat9'
from fp_inference import infer_beats
print(len(r_peaks))
res = infer_beats(sig, peaks, record_name, fs)

<class 'numpy.ndarray'>
108843
max_beats=300000
Mortara/pat9: 83199 usable beats of 83199 given
83199 beats clustered in 20-D PCA space, no reference labels (ARI / purity not computed)
mcs=3

method    clusters  noise  silhouette     ARI  purity
kmeans           2      0       0.302     -       -  
hdbscan         25  11458      -0.014     -       -  

Saved to c:\Users\Artin\Desktop\ecg_hierarchical_v4_pipeline\inference\Mortara\pat9


In [60]:
import fp_inference
import importlib
d = np.load("inference/Mortara/pat9/clusters/hdbscan/cluster1/cluster1_rpeaks.npz")
peaks = d["r_peaks"]  
print(type(peaks))
importlib.reload(fp_inference)
record_name='Mortara/pat11'
from fp_inference import infer_beats
print(len(r_peaks))
res = infer_beats(sig, peaks, record_name, fs)

<class 'numpy.ndarray'>
108843
max_beats=300000
Mortara/pat11: 71660 usable beats of 71660 given
71660 beats clustered in 8-D PCA space, no reference labels (ARI / purity not computed)
mcs=3

method    clusters  noise  silhouette     ARI  purity
kmeans           2      0       0.379     -       -  
hdbscan       3403  52478       0.227     -       -  

Saved to c:\Users\Artin\Desktop\ecg_hierarchical_v4_pipeline\inference\Mortara\pat11


In [11]:
print(len(sig))

31098600


In [4]:
"""
ECG beat clustering  (robust version)
-------------------------------------
Rule: two beats belong to the same cluster if
        sum((b - median)^2)  <  0.05 * min(sum(b^2), sum(median^2))
A new beat is compared to the CURRENT MEDIAN of every cluster.
Best (smallest-ratio) matching cluster wins; if none match, a new cluster is created.

Plots: black-only. Each cluster PNG shows all its beats in shallow black
and the cluster median as a darker, thicker black line.
"""

import os
import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless-safe
import matplotlib.pyplot as plt


# ======================================================================
# 1) Robust beat extraction
# ======================================================================
def _normalize_rpeaks(r_peaks, fs, signal_len, verbose=True):
    """
    Turn r_peaks into a clean 1-D int array of sample indices.
    Handles:  boolean mask, 2-D column/row, float seconds, float samples,
              list/array of anything numeric.
    """
    r = np.asarray(r_peaks)

    # boolean mask -> indices
    if r.dtype == bool:
        if r.size != signal_len:
            raise ValueError(
                f"Boolean r_peaks has length {r.size} but signal length is "
                f"{signal_len}. Cannot interpret as a mask.")
        r = np.flatnonzero(r)

    r = r.ravel()

    # if it's empty, nothing to do
    if r.size == 0:
        return r.astype(int)

    # Detect "seconds vs samples":
    #   - if dtype is float AND max value is tiny relative to signal_len,
    #     treat as seconds and multiply by fs.
    #   - also handle integer arrays that are clearly seconds (max << signal_len).
    rmax = float(np.max(r))
    looks_like_seconds = (rmax < signal_len / 10.0) and (rmax <= signal_len / fs * 1.5)
    # (the second condition: max seconds <= ~1.5 * recording duration)

    if looks_like_seconds:
        if verbose:
            print(f"[r_peaks] values look like SECONDS (max={rmax:.3f}); "
                  f"multiplying by fs={fs}.")
        r = np.round(r * fs)

    r = r.astype(int)

    if verbose:
        print(f"[r_peaks] n={r.size}, min={r.min()}, max={r.max()}, "
              f"signal_len={signal_len}")

    return r


def extract_beats(signal, r_peaks, fs, pre_ms=250, post_ms=300,
                  strict=True, verbose=True):
    """
    Cut fixed windows around R peaks.

    strict=True  : drop beats whose window falls partly outside the signal.
    strict=False : keep them, NaN-pad the out-of-range part.
    """
    sig = np.asarray(signal).ravel().astype(float)
    pre  = int(round(pre_ms  / 1000.0 * fs))
    post = int(round(post_ms / 1000.0 * fs))
    L    = pre + post

    r = _normalize_rpeaks(r_peaks, fs, len(sig), verbose=verbose)

    beats, valid_peaks = [], []
    skipped_low = skipped_high = 0

    for pk in r:
        s, e = pk - pre, pk + post
        if s < 0 or e > len(sig):
            if strict:
                if s < 0: skipped_low  += 1
                if e > len(sig): skipped_high += 1
                continue
            w = np.full(L, np.nan, dtype=float)
            s2, e2 = max(s, 0), min(e, len(sig))
            if e2 > s2:
                w[s2 - s : e2 - s] = sig[s2:e2]
            beats.append(w)
            valid_peaks.append(pk)
            continue
        beats.append(sig[s:e])
        valid_peaks.append(pk)

    if verbose:
        kept = len(beats)
        total = len(r)
        print(f"[extract_beats] kept {kept}/{total} beats "
              f"(pre={pre}, post={post}, L={L}); "
              f"skipped low={skipped_low}, high={skipped_high}")
        if kept == 0 and total > 0:
            print("  !! 0 beats kept. Check: fs correct? r_peaks in samples? "
                  "signal long enough?")
        elif kept < total and strict:
            print("  Note: some beats were dropped at the edges. "
                  "Use strict=False to NaN-pad them instead.")

    return np.asarray(beats, dtype=float), np.asarray(valid_peaks, dtype=int)


# ======================================================================
# 2) Clustering
# ======================================================================
def energy(x):
    return float(np.dot(x, x))


def cluster_beats(beats, threshold=0.05, verbose=True):
    """
    beats: (N, L) array
    Returns labels (N,), list of cluster dicts {'median':..., 'beats':[...]}
    """
    clusters = []
    labels = np.full(len(beats), -1, dtype=int)

    for i, beat in enumerate(beats):
        # skip beats that contain NaN (edge-padded, if strict=False)
        if not np.all(np.isfinite(beat)):
            continue

        e_beat = energy(beat)
        best_c, best_ratio = -1, np.inf

        for c, cl in enumerate(clusters):
            med   = cl['median']
            e_med = energy(med)
            e_err = energy(beat - med)

            denom = min(e_beat, e_med)
            ratio = e_err / denom if denom > 0 else np.inf

            if ratio < threshold and ratio < best_ratio:
                best_ratio, best_c = ratio, c

        if best_c == -1:
            clusters.append({'beats': [beat], 'median': beat.copy()})
            labels[i] = len(clusters) - 1
        else:
            cl = clusters[best_c]
            cl['beats'].append(beat)
            cl['median'] = np.median(np.vstack(cl['beats']), axis=0)
            labels[i] = best_c

    if verbose:
        sizes = [len(c['beats']) for c in clusters]
        print(f"[cluster_beats] formed {len(clusters)} clusters: {sizes}")

    return labels, clusters


# ======================================================================
# 3) Save each cluster to PNG  (black-only)
# ======================================================================
def save_clusters(clusters, fs, out_dir="RMS_Clusters",
                  pre_ms=250, post_ms=300):
    if len(clusters) == 0:
        print("[save_clusters] no clusters to plot.")
        return
    os.makedirs(out_dir, exist_ok=True)

    L    = len(clusters[0]['median'])
    pre  = int(round(pre_ms / 1000.0 * fs))
    t_ms = (np.arange(L) - pre) / fs * 1000.0

    for ci, cl in enumerate(clusters):
        beats  = np.vstack(cl['beats'])
        median = cl['median']
        n      = beats.shape[0]

        fig, ax = plt.subplots(figsize=(8, 4))

        for b in beats:
            ax.plot(t_ms, b, color="black", lw=0.5, alpha=0.15)
        ax.plot(t_ms, median, color="black", lw=2.0, alpha=1.0)

        ax.axvline(0, color="black", lw=0.6, ls=":", alpha=0.5)
        ax.set_xlabel("time relative to R peak (ms)")
        ax.set_ylabel("amplitude")
        ax.set_title(f"Cluster {ci}  —  {n} beat(s)")
        ax.grid(alpha=0.3)
        fig.tight_layout()

        fname = os.path.join(out_dir, f"cluster_{ci:03d}_n{n}.png")
        fig.savefig(fname, dpi=120)
        plt.close(fig)

    print(f"[save_clusters] saved {len(clusters)} PNGs to '{out_dir}/'")


# ======================================================================
# 4) Overview plot of all medians  (black-only)
# ======================================================================
def save_overview(clusters, fs, out_dir="RMS_Clusters",
                  pre_ms=250, post_ms=300):
    if len(clusters) == 0:
        return
    os.makedirs(out_dir, exist_ok=True)
    L    = len(clusters[0]['median'])
    pre  = int(round(pre_ms / 1000.0 * fs))
    t_ms = (np.arange(L) - pre) / fs * 1000.0

    fig, ax = plt.subplots(figsize=(9, 4.5))
    for cl in clusters:
        ax.plot(t_ms, cl['median'], color="black", lw=1.4, alpha=0.85)
    ax.axvline(0, color="black", lw=0.6, ls=":", alpha=0.5)
    ax.set_xlabel("time relative to R peak (ms)")
    ax.set_ylabel("amplitude")
    ax.set_title("Cluster medians — overview")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "00_overview_medians.png"), dpi=120)
    plt.close(fig)
    print(f"[save_overview] saved overview PNG to '{out_dir}/'")


# ======================================================================
# 5) Pipeline
# ======================================================================
def run(signal, r_peaks, fs,
        pre_ms=250, post_ms=300,
        threshold=0.05,
        out_dir="RMS_Clusters",
        strict=True,
        verbose=True):
    beats, valid_peaks = extract_beats(
        signal, r_peaks, fs,
        pre_ms=pre_ms, post_ms=post_ms,
        strict=strict, verbose=verbose)

    labels, clusters = cluster_beats(beats, threshold=threshold,
                                     verbose=verbose)

    save_clusters(clusters, fs, out_dir=out_dir,
                  pre_ms=pre_ms, post_ms=post_ms)
    save_overview(clusters, fs, out_dir=out_dir,
                  pre_ms=pre_ms, post_ms=post_ms)

    return labels, clusters, valid_peaks


# ======================================================================
# 6) Diagnostics (call standalone if needed)
# ======================================================================
def diagnose(signal, r_peaks, fs, pre_ms=250, post_ms=300):
    sig = np.asarray(signal).ravel()
    r   = np.asarray(r_peaks).ravel()
    pre  = int(round(pre_ms  / 1000.0 * fs))
    post = int(round(post_ms / 1000.0 * fs))

    print("--- diagnostics ---")
    print(f"signal len         : {len(sig)} samples "
          f"(≈ {len(sig)/fs:.2f} s at fs={fs})")
    print(f"r_peaks count      : {r.size}  dtype={np.asarray(r_peaks).dtype}")
    if r.size:
        print(f"r_peaks min / max  : {r.min()} / {r.max()}  "
              f"(signal max index = {len(sig)-1})")
        print(f"looks like seconds : "
              f"{r.max() < len(sig)/10 and r.max() <= len(sig)/fs*1.5}")
        if np.issubdtype(r.dtype, np.number):
            ri = r.astype(int)
            low  = int((ri - pre < 0).sum())
            high = int((ri + post > len(sig)).sum())
            print(f"skipped low side   : {low}")
            print(f"skipped high side  : {high}")
            print(f"valid windows      : "
                  f"{int(((ri - pre >= 0) & (ri + post <= len(sig))).sum())}")
    print("-------------------")


# ======================================================================


In [7]:
diagnose(sig, r_peaks, fs)                 # always run this first
labels, clusters, valid_peaks = run(
    sig, r_peaks, fs,
    pre_ms=250, post_ms=300,
    threshold=0.05,
    out_dir="RMS_Clusters",
)

--- diagnostics ---
signal len         : 15549300 samples (≈ 86385.00 s at fs=180)
r_peaks count      : 108843  dtype=int64
r_peaks min / max  : 7 / 15548354  (signal max index = 15549299)
looks like seconds : False
skipped low side   : 1
skipped high side  : 0
valid windows      : 108842
-------------------
[r_peaks] n=108843, min=7, max=15548354, signal_len=15549300
[extract_beats] kept 108842/108843 beats (pre=45, post=54, L=99); skipped low=1, high=0
  Note: some beats were dropped at the edges. Use strict=False to NaN-pad them instead.


KeyboardInterrupt: 

In [8]:
"""
ECG beat clustering  (streaming / interleaved version)
-------------------------------------------------------
Rule: two beats belong to the same cluster if
        sum((b - median)^2)  <  0.05 * min(sum(b^2), sum(median^2))
A new beat is compared to the CURRENT MEDIAN of every cluster.
Best (smallest-ratio) matching cluster wins; if none match, a new cluster is created.

Clustering and plotting are interleaved:
  - after EACH beat, the affected cluster's PNG is (re)written
  - the overview PNG is refreshed every `overview_every` beats
  - so you can watch 'RMS_Clusters/' update in a file browser in real time

Plots are black-only. Member beats: shallow black. Median: darker thick black.
"""

import os
import glob
import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless-safe
import matplotlib.pyplot as plt


# ======================================================================
# 1) Robust beat extraction
# ======================================================================
def _normalize_rpeaks(r_peaks, fs, signal_len, verbose=True):
    r = np.asarray(r_peaks)
    if r.dtype == bool:
        if r.size != signal_len:
            raise ValueError(
                f"Boolean r_peaks length {r.size} != signal length {signal_len}.")
        r = np.flatnonzero(r)
    r = r.ravel()
    if r.size == 0:
        return r.astype(int)
    rmax = float(np.max(r))
    looks_like_seconds = (rmax < signal_len / 10.0) and (rmax <= signal_len / fs * 1.5)
    if looks_like_seconds:
        if verbose:
            print(f"[r_peaks] looks like SECONDS (max={rmax:.3f}); x fs")
        r = np.round(r * fs)
    r = r.astype(int)
    if verbose:
        print(f"[r_peaks] n={r.size}, min={r.min()}, max={r.max()}, "
              f"signal_len={signal_len}")
    return r


def extract_beats(signal, r_peaks, fs, pre_ms=250, post_ms=300,
                  strict=True, verbose=True):
    sig  = np.asarray(signal).ravel().astype(float)
    pre  = int(round(pre_ms  / 1000.0 * fs))
    post = int(round(post_ms / 1000.0 * fs))
    L    = pre + post
    r    = _normalize_rpeaks(r_peaks, fs, len(sig), verbose=verbose)

    beats, valid_peaks = [], []
    skipped_low = skipped_high = 0
    for pk in r:
        s, e = pk - pre, pk + post
        if s < 0 or e > len(sig):
            if strict:
                if s < 0: skipped_low += 1
                if e > len(sig): skipped_high += 1
                continue
            w = np.full(L, np.nan)
            s2, e2 = max(s, 0), min(e, len(sig))
            if e2 > s2:
                w[s2 - s : e2 - s] = sig[s2:e2]
            beats.append(w); valid_peaks.append(pk); continue
        beats.append(sig[s:e]); valid_peaks.append(pk)

    if verbose:
        print(f"[extract_beats] kept {len(beats)}/{len(r)} beats "
              f"(pre={pre}, post={post}); skipped low={skipped_low}, "
              f"high={skipped_high}")

    return np.asarray(beats, dtype=float), np.asarray(valid_peaks, dtype=int)


# ======================================================================
# 2) Energy helper
# ======================================================================
def energy(x):
    return float(np.dot(x, x))


# ======================================================================
# 3) Plotting helpers  (black-only)
# ======================================================================
def _save_one_cluster(cl, ci, t_ms, out_dir):
    """Write/overwrite the PNG of a single cluster."""
    beats  = np.vstack(cl['beats'])
    median = cl['median']
    n      = beats.shape[0]

    fig, ax = plt.subplots(figsize=(8, 4))
    for b in beats:
        ax.plot(t_ms, b, color="black", lw=0.5, alpha=0.15)
    ax.plot(t_ms, median, color="black", lw=2.0, alpha=1.0)
    ax.axvline(0, color="black", lw=0.6, ls=":", alpha=0.5)
    ax.set_xlabel("time relative to R peak (ms)")
    ax.set_ylabel("amplitude")
    ax.set_title(f"Cluster {ci}  —  {n} beat(s)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    # Fixed filename -> the file is OVERWRITTEN as the cluster grows
    fig.savefig(os.path.join(out_dir, f"cluster_{ci:03d}.png"), dpi=120)
    plt.close(fig)


def _save_overview(clusters, t_ms, out_dir):
    """Write/overwrite the overview PNG of all cluster medians."""
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for cl in clusters:
        ax.plot(t_ms, cl['median'], color="black", lw=1.4, alpha=0.85)
    ax.axvline(0, color="black", lw=0.6, ls=":", alpha=0.5)
    ax.set_xlabel("time relative to R peak (ms)")
    ax.set_ylabel("amplitude")
    ax.set_title(f"Cluster medians — overview  ({len(clusters)} clusters)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "00_overview_medians.png"), dpi=120)
    plt.close(fig)


# ======================================================================
# 4) Streaming clustering + plotting
# ======================================================================
def cluster_and_plot_streaming(beats, fs,
                                threshold=0.05,
                                out_dir="RMS_Clusters",
                                pre_ms=250, post_ms=300,
                                save_every=1,
                                overview_every=20,
                                live_show=False,
                                verbose=True):
    """
    Cluster beats one-by-one and write/update PNGs incrementally.

    save_every     : write the affected cluster PNG every N beats
                     (1 = after every beat; raise to speed up on big records)
    overview_every : refresh overview PNG every N beats
    live_show      : also display an interactive window (needs a GUI backend)
    """
    os.makedirs(out_dir, exist_ok=True)

    L    = beats.shape[1]
    pre  = int(round(pre_ms / 1000.0 * fs))
    t_ms = (np.arange(L) - pre) / fs * 1000.0
    N    = len(beats)

    clusters = []
    labels   = np.full(N, -1, dtype=int)

    if live_show:
        plt.ion()
        live_fig, live_ax = plt.subplots(figsize=(8, 4))

    for i, beat in enumerate(beats):
        if not np.all(np.isfinite(beat)):
            if verbose:
                print(f"[stream] beat {i+1:>4}/{N}  ->  SKIP (NaN)")
            continue

        # ---- classify ----
        e_beat = energy(beat)
        best_c, best_ratio = -1, np.inf
        for c, cl in enumerate(clusters):
            med   = cl['median']
            e_med = energy(med)
            e_err = energy(beat - med)
            denom = min(e_beat, e_med)
            ratio = e_err / denom if denom > 0 else np.inf
            if ratio < threshold and ratio < best_ratio:
                best_ratio, best_c = ratio, c

        # ---- assign ----
        is_new = best_c == -1
        if is_new:
            clusters.append({'beats': [beat], 'median': beat.copy()})
            assigned = len(clusters) - 1
        else:
            cl = clusters[best_c]
            cl['beats'].append(beat)
            cl['median'] = np.median(np.vstack(cl['beats']), axis=0)
            assigned = best_c
        labels[i] = assigned

        if verbose:
            tag = "NEW" if is_new else f"#{assigned}"
            print(f"[stream] beat {i+1:>4}/{N}  ->  cluster {tag:<4} "
                  f"(size={len(clusters[assigned]['beats'])}, "
                  f"n_clusters={len(clusters)})")

        # ---- interleaved save ----
        do_save = ((i + 1) % save_every == 0) or (i == N - 1)
        if do_save:
            _save_one_cluster(clusters[assigned], assigned, t_ms, out_dir)
            if (i + 1) % overview_every == 0 or i == N - 1:
                _save_overview(clusters, t_ms, out_dir)

        # ---- optional live window ----
        if live_show:
            live_ax.clear()
            cl = clusters[assigned]
            for b in np.vstack(cl['beats']):
                live_ax.plot(t_ms, b, color="black", lw=0.5, alpha=0.15)
            live_ax.plot(t_ms, cl['median'], color="black", lw=2.0)
            live_ax.axvline(0, color="black", lw=0.6, ls=":", alpha=0.5)
            live_ax.set_title(f"beat {i+1}/{N} -> cluster {assigned} "
                              f"(n={len(cl['beats'])})")
            live_ax.grid(alpha=0.3)
            live_fig.tight_layout()
            live_fig.canvas.draw_idle()
            plt.pause(0.001)

    if live_show:
        plt.ioff()
        plt.show(block=True)

    if verbose:
        print(f"[stream] done: {len(clusters)} clusters, "
              f"sizes={[len(c['beats']) for c in clusters]}")

    return labels, clusters


# ======================================================================
# 5) Full pipeline (extract -> stream-cluster+plot)
# ======================================================================
def run(signal, r_peaks, fs,
        pre_ms=250, post_ms=300,
        threshold=0.05,
        out_dir="RMS_Clusters",
        strict=True,
        save_every=1,
        overview_every=20,
        live_show=False,
        clean=True,
        verbose=True):
    """
    Extract -> cluster & plot simultaneously.
    Returns (labels, clusters, valid_peaks).
    """
    os.makedirs(out_dir, exist_ok=True)
    if clean:
        removed = 0
        for f in glob.glob(os.path.join(out_dir, "*.png")):
            try:
                os.remove(f); removed += 1
            except OSError:
                pass
        if verbose and removed:
            print(f"[run] cleaned {removed} old PNG(s) from '{out_dir}/'")

    beats, valid_peaks = extract_beats(
        signal, r_peaks, fs, pre_ms=pre_ms, post_ms=post_ms,
        strict=strict, verbose=verbose)

    if len(beats) == 0:
        print("[run] no beats to cluster.")
        return np.array([], dtype=int), [], valid_peaks

    labels, clusters = cluster_and_plot_streaming(
        beats, fs,
        threshold=threshold, out_dir=out_dir,
        pre_ms=pre_ms, post_ms=post_ms,
        save_every=save_every, overview_every=overview_every,
        live_show=live_show, verbose=verbose)

    return labels, clusters, valid_peaks


# ======================================================================
# 6) Diagnostics
# ======================================================================
def diagnose(signal, r_peaks, fs, pre_ms=250, post_ms=300):
    sig = np.asarray(signal).ravel()
    r   = np.asarray(r_peaks).ravel()
    pre  = int(round(pre_ms  / 1000.0 * fs))
    post = int(round(post_ms / 1000.0 * fs))
    print("--- diagnostics ---")
    print(f"signal len        : {len(sig)} samples (≈ {len(sig)/fs:.2f} s)")
    print(f"r_peaks count     : {r.size}  dtype={np.asarray(r_peaks).dtype}")
    if r.size:
        print(f"r_peaks min/max   : {r.min()} / {r.max()}  "
              f"(signal max idx = {len(sig)-1})")
        if np.issubdtype(r.dtype, np.number):
            ri = r.astype(int)
            print(f"skipped low side  : {int((ri - pre < 0).sum())}")
            print(f"skipped high side : {int((ri + post > len(sig)).sum())}")
            print(f"valid windows     : "
                  f"{int(((ri - pre >= 0) & (ri + post <= len(sig))).sum())}")
    print("-------------------")



In [9]:
diagnose(sig, r_peaks, fs)

# cluster + plot simultaneously
labels, clusters, valid_peaks = run(
    sig, r_peaks, fs,
    pre_ms=250, post_ms=300,
    threshold=0.05,
    out_dir="RMS_Clusters",
    save_every=1,
    overview_every=10,
    clean=True,
)

--- diagnostics ---
signal len        : 15549300 samples (≈ 86385.00 s)
r_peaks count     : 108843  dtype=int64
r_peaks min/max   : 7 / 15548354  (signal max idx = 15549299)
skipped low side  : 1
skipped high side : 0
valid windows     : 108842
-------------------
[r_peaks] n=108843, min=7, max=15548354, signal_len=15549300
[extract_beats] kept 108842/108843 beats (pre=45, post=54); skipped low=1, high=0
[stream] beat    1/108842  ->  cluster NEW  (size=1, n_clusters=1)
[stream] beat    2/108842  ->  cluster NEW  (size=1, n_clusters=2)
[stream] beat    3/108842  ->  cluster #1   (size=2, n_clusters=2)
[stream] beat    4/108842  ->  cluster NEW  (size=1, n_clusters=3)
[stream] beat    5/108842  ->  cluster #1   (size=3, n_clusters=3)
[stream] beat    6/108842  ->  cluster #1   (size=4, n_clusters=3)
[stream] beat    7/108842  ->  cluster NEW  (size=1, n_clusters=4)
[stream] beat    8/108842  ->  cluster NEW  (size=1, n_clusters=5)
[stream] beat    9/108842  ->  cluster NEW  (size=1, n_c

PermissionError: [Errno 13] Permission denied: 'RMS_Clusters\\cluster_003.png'

In [14]:
print(r_peaks)
print(len(r_peaks))

[       7      169      348 ... 15548189 15548290 15548354]
108843


In [7]:
import numpy as np
import importlib
import ecg_vae_pipeline_v7
importlib.reload(ecg_vae_pipeline_v7)
from ecg_vae_pipeline_v7 import run_vae_clustering_pipeline, visualize_results


X_scaled, z_mean, labels, history = run_vae_clustering_pipeline(sig, r_peaks, fs, per_beat_norm=True)
visualize_results(X_scaled, z_mean, labels, fs, pre_ms=250)

R-peak polarity (auto): max
Extracted 6988 valid beats of length 99 samples after cleaning.
Building and training VAE...
Epoch 1/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - kl_loss: 4.1572 - loss: 27.0192 - reconstruction_loss: 22.8620
Epoch 2/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - kl_loss: 1.9126 - loss: 8.0100 - reconstruction_loss: 6.0974
Epoch 3/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - kl_loss: 1.5919 - loss: 7.2728 - reconstruction_loss: 5.6809
Epoch 4/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - kl_loss: 0.8338 - loss: 6.2023 - reconstruction_loss: 5.3685
Epoch 5/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - kl_loss: 0.4185 - loss: 5.2051 - reconstruction_loss: 4.7867
Epoch 6/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - kl_loss: 0.5485 - loss: 5.0401 - reconstruction_loss: 4.4916
Epoch 7/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - kl_loss: 0.5945 - loss: 4.9800 - reconstruction_loss: 4.3855
Epoch 8/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - kl_loss: 0.6172 - 

c:\Users\Artin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


Found 7 clusters (+ 1 noise cluster).
Generating and saving t-SNE projection to 'VAE_Clusters'...
Saving individual cluster waveforms to 'VAE_Clusters'...
Successfully saved all cluster plots in 'VAE_Clusters'.


In [15]:


print(f"Signal length: {len(sig)}")
print(f"Number of r_peaks: {len(r_peaks)}")
if len(r_peaks) > 0:
    print(f"First 5 peaks: {r_peaks[:5]}")
    print(f"Max peak index: {max(r_peaks)}")

Signal length: 15549300
Number of r_peaks: 108843
First 5 peaks: [  7 169 348 531 706]
Max peak index: 15548354


In [31]:
import matplotlib.pyplot

In [7]:
import matplotlib.pyplot as plt
# import importlib
# importlib.reload(matplotlib.pyplot)
%matplotlib tk
pre=int(0.25*fs)
post=int(0.3*fs)
for i in r_peaks:
    print(len(sig[i-pre:i+post]))
    plt.plot(sig[i-pre:i+post])
plt.savefig("p.png")
plt.show()

0
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99
99

In [ ]:
import importlib
import ecg_beat_clusteing
importlib.reload(ecg_beat_clustering)
from ecg_beat_clustering import ECGBeatClusterer

clusterer = ECGBeatClusterer(threshold=0.1)

clusterer.run(
    signal=sig,
    r_peaks=r_peaks,
    fs=fs
    save_folder="RMS_Cluster"
)


ECG CLUSTERING RESULTS
----------------------------------------
Sampling frequency: 360 Hz
Extracted beats:    651
Skipped R peaks:    2
Number of clusters: 173
Threshold:          0.0500 (5.00%)
Cluster 0: 1 beats
Cluster 1: 60 beats
Cluster 2: 20 beats
Cluster 3: 1 beats
Cluster 4: 74 beats
Cluster 5: 1 beats
Cluster 6: 1 beats
Cluster 7: 1 beats
Cluster 8: 1 beats
Cluster 9: 1 beats
Cluster 10: 1 beats
Cluster 11: 1 beats
Cluster 12: 1 beats
Cluster 13: 1 beats
Cluster 14: 1 beats
Cluster 15: 1 beats
Cluster 16: 107 beats
Cluster 17: 94 beats
Cluster 18: 1 beats
Cluster 19: 1 beats
Cluster 20: 1 beats
Cluster 21: 27 beats
Cluster 22: 63 beats
Cluster 23: 1 beats
Cluster 24: 5 beats
Cluster 25: 1 beats
Cluster 26: 1 beats
Cluster 27: 1 beats
Cluster 28: 1 beats
Cluster 29: 1 beats
Cluster 30: 4 beats
Cluster 31: 12 beats
Cluster 32: 1 beats
Cluster 33: 1 beats
Cluster 34: 1 beats
Cluster 35: 1 beats
Cluster 36: 1 beats
Cluster 37: 1 beats
Cluster 38: 1 beats
Cluster 39: 1 beats
Clus